# Subscription fatigue and adoption barriers: the funnel and what sits above it

**Claim cluster:** `C085`, `C086`, `C114`, `C055`  
**Source of truth:** `data/smarthome.duckdb` (read-only), via `src/viz_core.py`

In [ ]:
import os
import sys
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, clear_output, display

# Find the repo root by walking up until schemas/stage2.sql appears, so the
# notebook runs from notebooks/, from the repo root, or from anywhere else.
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "schemas" / "stage2.sql").exists():
        ROOT = _candidate
        break
else:  # pragma: no cover - only reachable outside a checkout
    raise FileNotFoundError("schemas/stage2.sql not found above " + str(_here))

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

# Option A topology: this wave reads one dedicated Stage 2 database. Point
# viz_core at it before import -- it resolves the path once at import time --
# so every fetch in this kernel uses the same file.
os.environ["SMARTHOME_DB"] = str(ROOT / "data" / "smarthome.duckdb")

import viz_core as viz  # noqa: E402

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

# Renderers matter for two reasons. "browser" (plotly's default) opens a
# window when a figure is shown outside a live kernel, which breaks headless
# execution. The bare "notebook" renderer in plotly 7 emits only text/html,
# carrying no figure payload, so a notebook could run clean and still show
# nothing. "plotly_mimetype+notebook" emits the figure JSON for JupyterLab and
# an inline HTML fallback elsewhere -- which is also what the headless test
# asserts on.
pio.renderers.default = "plotly_mimetype+notebook"

# Every figure in this notebook reads only from data/smarthome.duckdb (read-only).
CLAIM_IDS = ('C085', 'C086', 'C114', 'C055')
assert viz.DEFAULT_DB_PATH.endswith("smarthome.duckdb"), viz.DEFAULT_DB_PATH

claims = viz.fetch_claims(list(CLAIM_IDS))
sources = viz.sources_for_claims(list(CLAIM_IDS)) if CLAIM_IDS else pd.DataFrame()
cluster_metrics = (
    viz.claim_metrics_lookup(list(CLAIM_IDS)) if CLAIM_IDS else pd.DataFrame()
)

# Corpus-wide row counts, read once so a notebook can state the size of what it
# is drawing from without confusing a whole-corpus count with the size of its
# own (possibly empty) claim cluster.
CORPUS = viz.connect_db()
try:
    corpus_counts = {
        table: CORPUS.execute(f"SELECT COUNT(1) FROM {table}").fetchone()[0]
        for table in ("stage2_claim", "metric", "entity", "source_mirror",
                      "predicate", "extraction_decision")
    }
finally:
    CORPUS.close()

def provenance(claim_id):
    """Collapsible evidence panel for one claim in this cluster."""
    panel = viz.render_provenance_panel(claim_id)
    return widgets.VBox([panel])

def show(*items):
    """Display data frames, HTML, or figures, one per argument."""
    for item in items:
        display(item)

def confidence_legend():
    """Show the colour coding applied to confidence across all figures."""
    display(HTML(
        "<b>Confidence colour coding:</b> "
        f"<span style='color:{viz.CONFIDENCE_COLORS['high']}'>high</span> &middot; "
        f"<span style='color:{viz.CONFIDENCE_COLORS['medium']}'>medium</span> &middot; "
        f"<span style='color:{viz.CONFIDENCE_COLORS['low']}'>low</span>"
    ))

## Provenance

Adoption and business numbers in this corpus are market-research figures with named but sometimes unlinked sources. The panels below record which source each claim rests on, and the gaps section flags the ones the extraction log already flagged as uncited.

In [ ]:
for claim_id in CLAIM_IDS:
    display(HTML(f"<h4>{claim_id} &mdash; {claims.set_index('local_id').loc[claim_id, 'claim_text']}</h4>"))
    display(provenance(claim_id))
confidence_legend()

## Primary evidence

### Subscription and ownership rates, with their denominators

Each bar is a metric the cluster cites, read from the database. The denominator of each rate is stated in its own `unit` cell and is carried into the bar label and the hover, because the corpus does not record a single shared population: 52% is a share of respondents, 53% a share of device owners, 19% a share of device owners, and 52% in `M019` a rate of DIY users hitting a setup problem. They are drawn side by side as stated rates, not multiplied into a conversion funnel, which would invent a population relationship the corpus never recorded.

In [ ]:
# (metric id, label, denominator note). The denominator is transcribed from the
# metric's unit cell, not invented; where the unit does not name a population,
# that is stated rather than guessed.
FUNNEL_SPECS = [
    ("M010", "Own a smart speaker or display", "share of respondents"),
    ("M031", "Do not pay a device subscription", "share of device owners"),
    ("M009", "Subscribe to video security", "share of smart home device owners"),
    ("M019", "Hit a setup or connectivity problem", "rate among DIY users"),
]
funnel_ids = [mid for mid, _, _ in FUNNEL_SPECS]
_raw = viz.metrics_by_id(funnel_ids)
_by_id = _raw.set_index("local_id", drop=False)

funnel = pd.DataFrame([
    {
        "stage": label,
        "metric": mid,
        "pct": viz.numeric_value(_by_id.loc[mid, "value"]) if mid in _by_id.index else None,
        "stated": _by_id.loc[mid, "value"] if mid in _by_id.index else None,
        "unit": _by_id.loc[mid, "unit"] if mid in _by_id.index else None,
        "denominator": denominator,
        "source": _by_id.loc[mid, "source_ref"] if mid in _by_id.index else None,
        "confidence": _by_id.loc[mid, "confidence"] if mid in _by_id.index else None,
    }
    for mid, label, denominator in FUNNEL_SPECS
])
funnel = funnel.dropna(subset=["pct"])
show(funnel)

fig = go.Figure(go.Bar(
    x=funnel["pct"],
    y=funnel["stage"] + " (" + funnel["metric"] + ")",
    orientation="h",
    marker=dict(color=[viz.CONFIDENCE_COLORS.get(c, viz.CONFIDENCE_COLORS[None])
                       for c in funnel["confidence"]]),
    customdata=list(zip(funnel["stated"], funnel["denominator"], funnel["source"],
                        funnel["confidence"], funnel["metric"])),
    hovertemplate=("%{y}<br>stated: %{customdata[0]}%<br>denominator: %{customdata[1]}"
                   "<br>source: %{customdata[2]} (metric %{customdata[4]})"
                   "<br>confidence: %{customdata[3]}<extra></extra>"),
))
fig.update_layout(
    title="Stated rates, each against its own denominator (not a conversion funnel)",
    xaxis_title="percent", height=340, yaxis=dict(automargin=True),
)
fig.show()

## Supporting context

### Two comparable churn figures, and why they are not one number

The corpus records residential monitoring cancellation at 11-14% per quarter (`M032`) and an industry annualised figure of 13% (`M033`), while Arlo's own reported churn is 1.0% per month (`M042`) against a 2.0-8.7% monthly streaming comparison (`M048`). These are different populations measured on different clocks; the chart below keeps them separate and labels the clock on each bar rather than converting them to a single headline rate.

In [ ]:
# Churn figures, kept apart because the corpus records them on three different
# clocks. The clock is transcribed from each metric's unit cell.
CHURN_IDS = ["M031", "M032", "M033", "M042", "M048"]
churn_raw = viz.metrics_by_id(CHURN_IDS)
display(HTML("<h4>Churn, as recorded</h4>"))
show(churn_raw[["local_id", "metric_name", "value", "unit", "confidence",
                "source_ref", "section"]].rename(
    columns={"local_id": "metric", "metric_name": "name", "unit": "stated_unit",
             "source_ref": "source", "confidence": "conf"}))

churn = viz.range_frame(CHURN_IDS)
show(churn[["metric", "name", "value", "low", "high", "unit", "confidence"]])

_records = churn.to_dict("records")
fig = go.Figure(go.Bar(
    x=[(r["low"] + r["high"]) / 2 for r in _records],
    y=[f"{r['metric']} {r['name']} ({r['unit']})" for r in _records],
    orientation="h",
    marker=dict(color=[viz.CONFIDENCE_COLORS.get(c, viz.CONFIDENCE_COLORS[None])
                       for c in churn["confidence"]]),
    error_x=dict(type="data",
                 array=[(r["high"] - r["low"]) / 2 for r in _records],
                 arrayminus=[(r["high"] - r["low"]) / 2 for r in _records],
                 visible=True),
    customdata=list(zip(churn["value"], churn["unit"], churn["confidence"])),
    hovertemplate=("%{y}<br>stated: %{customdata[0]}<br>per: %{customdata[1]}"
                   "<br>confidence: %{customdata[2]}<extra></extra>"),
))
fig.update_layout(
    title="Churn figures, each on its own measurement clock (not converted)",
    xaxis_title="percent per stated period", height=340, yaxis=dict(automargin=True),
)
fig.show()

display(HTML("<h4>Arlo unit economics recorded in the corpus</h4>"))
econ = viz.metrics_by_id(["M044", "M045", "M046", "M040", "M043"])
show(econ[["local_id", "metric_name", "value", "unit", "confidence",
           "source_ref"]].rename(
    columns={"local_id": "metric", "metric_name": "name", "source_ref": "source",
             "confidence": "conf"}))

## Uncertainty and gaps

### The rates above are not a funnel

The bar chart above deliberately does not narrow into a funnel. The four rates are stated against three different populations (all respondents, all device owners, DIY users) drawn from different sources, and `M019` is a problem rate rather than a stage. Multiplying them into an end-to-end conversion would assert a population relationship the corpus never recorded. Each denominator is carried in the bar label and the hover instead.

- **`M032` and `M048` are not comparable as drawn.** 11-14% per quarter of monitoring subscribers is a different population on a different clock from Arlo's 1.0% per month; converting one into the other would be arithmetic, not measurement, so the bar labels carry the clock instead.
- **`C055` is in this cluster for its evidence shape, not its subject.** It is an `inference` at high confidence and is the only energy claim here; it is kept because it is the corpus's own model discipline (weakest economic link = battery storage) and shows what a high-confidence inference looks like next to a documented fact.
- **Sources S56-S60 and S61-S80 have no URL.** The extraction log records this in `L047` and `L048`; those barriers and market figures cannot be checked against a primary publication from this database.
- **No time series.** Every figure is a point observation; the corpus holds no adoption or churn trend, so no line chart is drawn.

In [ ]:
display(HTML("<h4>Claims in this cluster with no falsifier</h4>"
             "<p>A missing falsifier means the corpus recorded no observation "
             "that would disconfirm the claim. The panel above flags these "
             "claims in <code>no_falsifier</code>.</p>"))
show(claims.loc[claims["falsifier_missing"], ["local_id", "claim_type",
                                              "confidence"]]
     .rename(columns={"local_id": "claim", "claim_type": "type",
                      "confidence": "conf"}))

display(HTML("<h4>Metrics with conditions not stated in the source</h4>"))
_gaps = cluster_metrics.loc[~cluster_metrics["conditions_stated"].fillna(False)]
if _gaps.empty:
    display(HTML("<p>No unstated conditions among the cluster's metrics.</p>"))
else:
    show(_gaps[["local_id", "metric_name", "value", "unit"]].rename(
        columns={"local_id": "metric", "metric_name": "name"}))

display(HTML("<h4>Extraction decisions that changed or qualified this material</h4>"))
_decisions = pd.DataFrame(
    viz.connect_db().execute(
        "SELECT local_id, decision_type, description, resolution "
        "FROM extraction_decision "
        "WHERE decision_type IN ('evidence_downgrade', 'classification_conflict', "
        "                           'falsifier_absent', 'condition_absent', "
        "                           'boundary_absent', 'scope_boundary', "
        "                           'omission') "
        "ORDER BY local_id"
    ).fetchall(),
    columns=["decision", "type", "description", "resolution"],
)
show(_decisions)

## Reproducibility

Metric values are read from `data/smarthome.duckdb` by local id and parsed with `viz.numeric_value` / `viz.range_value`. The measurement clock shown on each churn bar is a notebook-level annotation, because the clock is part of the metric's `unit` cell in the corpus and is not a separate field.

In [ ]:
# The exact queries behind every figure above, re-run verbatim.
import duckdb
con = duckdb.connect(str(ROOT / 'data' / 'smarthome.duckdb'), read_only=True)
print('database:', con.execute('SELECT current_database()').fetchone()[0])
for table in ('stage2_claim', 'metric', 'entity', 'source_mirror',
               'predicate', 'workflow_stage', 'extraction_decision'):
    total = con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'{table:22} {total:>6} rows')
con.close()